In [2]:
# -*- coding: utf-8 -*-
"""
PyTorch GPU 论文实验（最终版）：
仅使用 direct_multi 模式：
- BaseLSTM：基础 LSTM 基线模型
- FeatureAttention_LSTM：带特征注意力的 LSTM 模型
直接预测 [t+1, t+2, t+3] 三个交易日的收盘价
"""

from __future__ import annotations

import copy
import json
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Any, Tuple

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    import akshare as ak
except Exception:
    ak = None

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 全局设置
# ============================================================

SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT_DIR = Path("thesis_25stocks_direct_multi_final")
ROOT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "20160101"
END_DATE = "20250701"

LOOKBACK = 90
FORECAST_HORIZONS = 3
TEST_RATIO = 0.20
VAL_RATIO_WITHIN_TRAIN = 0.20

ARCH_NAMES = ["BaseLSTM", "FeatureAttention_LSTM"]

STOCK_POOL = [
    {"symbol": "600036", "name": "招商银行", "sector": "银行"},
    {"symbol": "000001", "name": "平安银行", "sector": "银行"},
    {"symbol": "600030", "name": "中信证券", "sector": "券商"},
    {"symbol": "601318", "name": "中国平安", "sector": "保险"},
    {"symbol": "600519", "name": "贵州茅台", "sector": "白酒消费"},
    {"symbol": "000858", "name": "五粮液", "sector": "白酒消费"},
    {"symbol": "600887", "name": "伊利股份", "sector": "食品饮料"},
    {"symbol": "603288", "name": "海天味业", "sector": "调味品"},
    {"symbol": "002594", "name": "比亚迪", "sector": "新能源汽车"},
    {"symbol": "300750", "name": "宁德时代", "sector": "动力电池"},
    {"symbol": "601633", "name": "长城汽车", "sector": "汽车"},
    {"symbol": "000625", "name": "长安汽车", "sector": "汽车"},
    {"symbol": "600031", "name": "三一重工", "sector": "工程机械"},
    {"symbol": "000338", "name": "潍柴动力", "sector": "工程机械"},
    {"symbol": "601668", "name": "中国建筑", "sector": "建筑"},
    {"symbol": "600276", "name": "恒瑞医药", "sector": "医药"},
    {"symbol": "000538", "name": "云南白药", "sector": "医药"},
    {"symbol": "300760", "name": "迈瑞医疗", "sector": "医疗器械"},
    {"symbol": "600309", "name": "万华化学", "sector": "化工"},
    {"symbol": "000651", "name": "格力电器", "sector": "家电"},
    {"symbol": "600690", "name": "海尔智家", "sector": "家电"},
    {"symbol": "002415", "name": "海康威视", "sector": "电子"},
    {"symbol": "000725", "name": "京东方A", "sector": "面板"},
    {"symbol": "002475", "name": "立讯精密", "sector": "消费电子"},
    {"symbol": "601899", "name": "紫金矿业", "sector": "有色金属"},
]

PAPER_STOCK_SYMBOL = "600519"


# ============================================================
# 基础设置
# ============================================================

def set_seed(seed: int = SEED):
    """设置随机种子，尽量提高实验的可复现性，方便在阅读主流程时快速理解它的职责。"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def setup_matplotlib_chinese():
    """配置中文字体和绘图的默认显示参数，方便在阅读主流程时快速理解它的职责。"""
    candidate_fonts = [
        "SimHei",
        "Microsoft YaHei",
        "STHeiti",
        "PingFang SC",
        "Noto Sans CJK SC",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    matplotlib.rcParams["font.sans-serif"] = candidate_fonts
    matplotlib.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.figsize"] = (12, 5)
    plt.rcParams["savefig.dpi"] = 180


set_seed(SEED)
setup_matplotlib_chinese()


# ============================================================
# 工具函数
# ============================================================

def check_dependencies():
    """检查关键依赖是否已安装，缺失时给出安装提示，方便在阅读主流程时快速理解它的职责。"""
    missing = []
    if ak is None:
        missing.append("akshare")
    if missing:
        raise ImportError(
            f"Missing packages: {missing}\n"
            "Please install:\n"
            "pip install akshare pandas numpy matplotlib scikit-learn torch joblib"
        )


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """计算回归任务常用的误差指标和 R2，方便在阅读主流程时快速理解它的职责。"""
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MSE": float(mse),
        "RMSE": float(np.sqrt(mse)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def count_parameters(model: nn.Module) -> int:
    """统计模型中可训练参数的总量，方便在阅读主流程时快速理解它的职责。"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_prediction(output: Any) -> torch.Tensor:
    """统一提取模型输出中的预测张量，这样主流程可以更统一地获取所需对象或结果。"""
    return output[0] if isinstance(output, tuple) else output


def safe_mkdir(path: Path):
    """安全创建目录；目录已存在时不报错，方便在阅读主流程时快速理解它的职责。"""
    path.mkdir(parents=True, exist_ok=True)


# ============================================================
# 技术指标
# ============================================================

def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    """根据收盘价序列计算 RSI 指标，结果会继续作为后续特征或评估指标的一部分。"""
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / (avg_loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(close: pd.Series,
                 fast: int = 12,
                 slow: int = 26,
                 signal: int = 9) -> Tuple[pd.Series, pd.Series, pd.Series]:
    """根据收盘价序列计算 MACD 及其相关量，结果会继续作为后续特征或评估指标的一部分。"""
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    macd_diff = macd - macd_signal
    return macd, macd_signal, macd_diff


# ============================================================
# 数据抓取
# ============================================================

def fetch_single_stock_data(symbol: str,
                            start_date: str = START_DATE,
                            end_date: str = END_DATE) -> pd.DataFrame:
    """抓取单只股票数据并统一成后续流程可用的格式，让后面的特征工程、训练和评估都能直接复用这一份标准化结果。"""
    # 先确认数据抓取依赖存在，再向行情接口请求原始数据。
    check_dependencies()

    # 这里使用 AkShare 的日线历史接口获取指定股票在时间区间内的数据。
    df = ak.stock_zh_a_hist(
        symbol=symbol,
        period="daily",
        start_date=start_date,
        end_date=end_date,
        adjust=""
    )

    # 如果接口没有返回有效数据，就立即抛出异常，避免后续流程继续出错。
    if df is None or len(df) == 0:
        raise RuntimeError(f"Failed to fetch data for symbol {symbol}")

    # 把原始中文列名统一转换成内部使用的标准字段名。
    df = df.rename(columns={
        "日期": "trade_date",
        "开盘": "open",
        "最高": "high",
        "最低": "low",
        "收盘": "close",
        "成交量": "volume",
        "成交额": "amount",
        "换手率": "turnover",
    })

    # 检查关键字段是否齐全，确保后续特征工程有完整输入。
    required = ["trade_date", "open", "high", "low", "close", "volume", "amount"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{symbol} missing required columns: {missing}")

    if "turnover" not in df.columns:
        df["turnover"] = np.nan

    # 只保留核心行情字段，并把日期和数值类型整理成统一格式。
    keep_cols = ["trade_date", "open", "high", "low", "close", "volume", "amount", "turnover"]
    df = df[keep_cols].copy()
    df["trade_date"] = pd.to_datetime(df["trade_date"])

    for col in ["open", "high", "low", "close", "volume", "amount", "turnover"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("trade_date").reset_index(drop=True)
    df = df.dropna(subset=["trade_date", "close"]).reset_index(drop=True)
    return df


# ============================================================
# 特征工程
# ============================================================

def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """基于原始行情数据构造技术指标和衍生特征，方便后续训练、验证或评估环节直接使用。"""
    out = df.copy()
    eps = 1e-8

    # 先构造收益率和对数收益率，描述价格的基本变化。
    out["return_1"] = out["close"].pct_change()
    out["log_return_1"] = np.log((out["close"] + eps) / (out["close"].shift(1) + eps))

    # 再加入不同窗口的均线以及收盘价相对均线的位置特征。
    out["ma_5"] = out["close"].rolling(5).mean()
    out["ma_10"] = out["close"].rolling(10).mean()
    out["ma_20"] = out["close"].rolling(20).mean()

    out["close_ma5_ratio"] = out["close"] / (out["ma_5"] + eps)
    out["close_ma10_ratio"] = out["close"] / (out["ma_10"] + eps)
    out["close_ma20_ratio"] = out["close"] / (out["ma_20"] + eps)

    # 继续补充波动率和动量特征，帮助模型感知短中期趋势。
    out["rolling_std_5"] = out["return_1"].rolling(5).std()
    out["rolling_std_10"] = out["return_1"].rolling(10).std()

    out["momentum_3"] = out["close"] / (out["close"].shift(3) + eps) - 1.0
    out["momentum_5"] = out["close"] / (out["close"].shift(5) + eps) - 1.0
    out["momentum_10"] = out["close"] / (out["close"].shift(10) + eps) - 1.0

    # 最后加入 RSI、MACD 等技术指标，进一步丰富输入信息。
    out["rsi_14"] = compute_rsi(out["close"], window=14)

    macd, macd_signal, macd_diff = compute_macd(out["close"])
    out["macd"] = macd
    out["macd_signal"] = macd_signal
    out["macd_diff"] = macd_diff

    # 统一清理无穷值和缺失值，保证输出特征表可以直接用于建模。
    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


# ============================================================
# 序列样本构造
# ============================================================

def build_multi_horizon_dataset(feature_df: pd.DataFrame,
                                lookback: int = LOOKBACK,
                                horizons: int = FORECAST_HORIZONS):
    """把时间序列切成过去窗口输入和未来多步标签，这样后续主流程就可以通过统一接口直接复用。"""
    # 先判断有效样本量是否足够，否则无法构造完整的输入窗口和预测标签。
    if len(feature_df) < lookback + horizons:
        raise ValueError("Not enough rows to build multi-horizon dataset.")

    # 拆出特征列、收盘价和日期，为后续滑窗构造做准备。
    feature_cols = [c for c in feature_df.columns if c != "trade_date"]
    values = feature_df[feature_cols].astype(float).values
    closes = feature_df["close"].astype(float).values
    dates = feature_df["trade_date"].values

    X_list, Y_list, D_list = [], [], []

    # 通过滑动时间窗口，把连续序列逐条切成监督学习样本。
    for end_idx in range(lookback - 1, len(feature_df) - horizons):
        start_idx = end_idx - lookback + 1
        X = values[start_idx:end_idx + 1]
        Y = closes[end_idx + 1:end_idx + horizons + 1]
        D = dates[end_idx + 1:end_idx + horizons + 1]

        X_list.append(X)
        Y_list.append(Y)
        D_list.append(D)

    # 把列表形式的数据统一转成数组，便于后续直接喂给模型。
    X = np.asarray(X_list, dtype=np.float32)
    Y = np.asarray(Y_list, dtype=np.float32)
    D = np.asarray(D_list)

    return X, Y, D, feature_cols


def chronological_split_indices(n: int, test_ratio: float = TEST_RATIO):
    """按时间顺序切分训练集和测试集索引，方便在阅读主流程时快速理解它的职责。"""
    split_idx = int(n * (1 - test_ratio))
    train_idx = np.arange(0, split_idx)
    test_idx = np.arange(split_idx, n)
    return train_idx, test_idx


# ============================================================
# 数据集与 DataLoader
# ============================================================

class SequenceDataset(Dataset):
    """把时序特征和标签封装成 PyTorch 数据集，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        """接收 numpy 数组并转换成张量形式保存，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        """返回数据集中样本的数量，这样 DataLoader 就能知道数据集中一共有多少条样本。"""
        return len(self.X)

    def __getitem__(self, idx: int):
        """按索引取出一条输入序列和对应标签，这样 DataLoader 取样时就能拿到一条完整的输入与标签。"""
        return self.X[idx], self.y[idx]


def create_train_val_loaders(X_train: np.ndarray,
                             y_train: np.ndarray,
                             batch_size: int = 32,
                             val_ratio: float = VAL_RATIO_WITHIN_TRAIN):
    """从训练集中按时间顺序划分验证集并构造 DataLoader，方便后续训练、验证或评估环节直接使用。"""
    # 按照时间顺序从训练集尾部切出验证集，避免随机打乱时序。
    n = len(X_train)
    val_start = int(n * (1 - val_ratio))

    X_tr, X_val = X_train[:val_start], X_train[val_start:]
    y_tr, y_val = y_train[:val_start], y_train[val_start:]

    # 把训练集和验证集分别封装成数据集对象。
    train_ds = SequenceDataset(X_tr, y_tr)
    val_ds = SequenceDataset(X_val, y_val)

    pin_memory = torch.cuda.is_available()

    # 再构造 DataLoader，后续训练和验证都按顺序读取样本。
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=pin_memory
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=pin_memory
    )
    return train_loader, val_loader


def create_test_loader(X_test: np.ndarray, y_test: np.ndarray, batch_size: int = 32):
    """把测试集包装成按顺序读取的 DataLoader，方便后续训练、验证或评估环节直接使用。"""
    test_ds = SequenceDataset(X_test, y_test)
    pin_memory = torch.cuda.is_available()
    return DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=pin_memory
    )


# ============================================================
# 模型定义
# ============================================================

class FeatureAttention(nn.Module):
    """根据输入特征生成逐维注意力权重，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(self, input_dim: int):
        """初始化特征注意力门控网络，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        inner = max(input_dim // 2, 8)
        self.gate = nn.Sequential(
            nn.Linear(input_dim, inner),
            nn.ReLU(),
            nn.Linear(inner, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor):
        """为输入特征生成权重，并输出加权后的结果，输入数据会在这里按照当前模块的设计完成主要计算。"""
        weights = self.gate(x)
        return x * weights, weights


class BaseLSTM(nn.Module):
    """基础 LSTM 模型，直接输出未来多步收盘价，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(self,
                 input_dim: int,
                 hidden_dim: int = 64,
                 num_layers: int = 1,
                 dropout: float = 0.2,
                 out_dim: int = 3):
        """初始化基础 LSTM 预测网络，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        """以前序时间窗口为输入，输出未来多步预测值，输入数据会在这里按照当前模块的设计完成主要计算。"""
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        pred = self.fc(self.dropout(last))
        return pred


class FeatureAttentionLSTM(nn.Module):
    """先做特征注意力，再用 LSTM 进行多步预测，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(self,
                 input_dim: int,
                 hidden_dim: int = 64,
                 num_layers: int = 1,
                 dropout: float = 0.2,
                 out_dim: int = 3):
        """初始化带特征注意力的 LSTM 网络，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        self.feature_attn = FeatureAttention(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        """先做特征注意力，再输出预测值和平均注意力，输入数据会在这里按照当前模块的设计完成主要计算。"""
        z, feat_weights = self.feature_attn(x)
        out, _ = self.lstm(z)
        last = out[:, -1, :]
        pred = self.fc(self.dropout(last))
        return pred, feat_weights.mean(dim=1)


# ============================================================
# 训练配置
# ============================================================

@dataclass
class TrainConfig:
    """训练超参数配置，用于统一管理搜索空间，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    hidden_dim: int = 64
    num_layers: int = 1
    dropout: float = 0.2
    learning_rate: float = 1e-3
    batch_size: int = 32
    max_epochs: int = 100
    patience: int = 12
    weight_decay: float = 1e-4


def get_model_search_space(model_name: str) -> List[TrainConfig]:
    """返回基础实验使用的超参数搜索空间，这样主流程可以更统一地获取所需对象或结果。"""
    return [
        TrainConfig(hidden_dim=64, num_layers=1, dropout=0.2, learning_rate=1e-3, batch_size=32),
        TrainConfig(hidden_dim=96, num_layers=1, dropout=0.2, learning_rate=5e-4, batch_size=32),
        TrainConfig(hidden_dim=96, num_layers=2, dropout=0.3, learning_rate=5e-4, batch_size=32),
    ]


def build_model(model_name: str,
                input_dim: int,
                cfg: TrainConfig,
                out_dim: int = 3) -> nn.Module:
    """根据模型名称构造对应的网络实例，这样后续主流程就可以通过统一接口直接复用。"""
    if model_name == "BaseLSTM":
        return BaseLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "FeatureAttention_LSTM":
        return FeatureAttentionLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    raise ValueError(f"Unknown model name: {model_name}")


def build_optimizer_scheduler_criterion(model: nn.Module, cfg: TrainConfig):
    """构建优化器、学习率调度器和损失函数，这样后续主流程就可以通过统一接口直接复用。"""
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=4
    )
    criterion = nn.HuberLoss(delta=1.0)
    return optimizer, scheduler, criterion


def run_epoch(model: nn.Module,
              loader: DataLoader,
              criterion: nn.Module,
              optimizer: Any = None):
    """执行一个训练轮次或验证轮次，并汇总损失与预测结果，便于后续统一计算指标和观察训练变化。"""
    # 根据是否传入优化器，判断当前轮次是训练还是验证。
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_n = 0
    preds, labels = [], []

    # 逐批读取数据，并把输入和标签移动到设定好的设备上。
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        # 完成前向计算，得到当前批次的预测结果和损失。
        out = model(X_batch)
        pred = get_prediction(out)
        loss = criterion(pred, y_batch)

        # 只有训练阶段才执行梯度清零、反向传播和参数更新。
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # 把每个批次的损失和预测结果累计起来，供轮次结束后统一统计。
        total_loss += loss.item() * len(y_batch)
        total_n += len(y_batch)

        preds.append(pred.detach().cpu().numpy())
        labels.append(y_batch.detach().cpu().numpy())

    # 轮次结束后计算平均损失，并拼接所有批次的预测与标签。
    avg_loss = total_loss / max(total_n, 1)
    preds = np.concatenate(preds, axis=0)
    labels = np.concatenate(labels, axis=0)
    return avg_loss, preds, labels


def train_with_early_stopping(model: nn.Module,
                              train_loader: DataLoader,
                              val_loader: DataLoader,
                              cfg: TrainConfig,
                              verbose: bool = True):
    """执行带早停机制的训练过程，并记录训练历史，用验证集表现来决定何时停止以及保留哪组最优权重。"""
    # 先搭建训练过程需要的优化器、调度器和损失函数。
    optimizer, scheduler, criterion = build_optimizer_scheduler_criterion(model, cfg)

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "lr": []}

    # 按轮次反复执行训练和验证，并持续记录历史指标。
    for epoch in range(cfg.max_epochs):
        train_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss, _, _ = run_epoch(model, val_loader, criterion, optimizer=None)

        # 根据验证集损失调整学习率，让训练过程更稳定。
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        # 一旦验证表现变好，就保存当前最优模型参数。
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # 按固定频率打印日志，方便观察损失变化趋势。
        if verbose and (epoch < 5 or (epoch + 1) % 10 == 0):
            print(
                f"    Epoch [{epoch+1:03d}/{cfg.max_epochs}] "
                f"train={train_loss:.6f} val={val_loss:.6f} lr={current_lr:.6g}"
            )

        # 若连续多轮没有提升，则触发提前停止，避免无效训练。
        if epochs_no_improve >= cfg.patience:
            if verbose:
                print(f"    提前停止于第 {epoch + 1} 轮")
            break

    # 训练结束后恢复到验证集表现最好的那组权重。
    model.load_state_dict(best_state)
    return model, history, float(best_val_loss)


def tune_model(model_name: str,
               X_train: np.ndarray,
               y_train: np.ndarray,
               input_dim: int,
               out_dim: int,
               verbose: bool = True):
    """遍历搜索空间，选出验证集表现最好的配置，从而为当前模型确定更稳妥的超参数组合。"""
    # 先取出当前模型要尝试的超参数配置列表。
    configs = get_model_search_space(model_name)
    best_result = None

    print("\n" + "=" * 72)
    print(f"调参 {model_name}")
    print("=" * 72)

    # 逐个配置进行训练，比较它们在验证集上的表现。
    for i, cfg in enumerate(configs, 1):
        print(f"\n  试验 {i}/{len(configs)} -> {cfg}")

        # 根据当前配置构造模型，并准备对应的数据加载器。
        model = build_model(model_name, input_dim, cfg, out_dim=out_dim)
        train_loader, val_loader = create_train_val_loaders(
            X_train, y_train, batch_size=cfg.batch_size, val_ratio=VAL_RATIO_WITHIN_TRAIN
        )

        # 运行带早停的训练流程，得到该配置下的最佳结果。
        model, history, best_val_loss = train_with_early_stopping(
            model, train_loader, val_loader, cfg, verbose=verbose
        )

        result = {
            "config": cfg,
            "history": history,
            "best_val_loss": best_val_loss,
            "state_dict": copy.deepcopy(model.state_dict()),
        }

        # 始终只保留验证损失最低的配置，作为最终候选。
        if best_result is None or best_val_loss < best_result["best_val_loss"]:
            best_result = result

    return best_result


# ============================================================
# 标准化与评估
# ============================================================

def fit_transform_X(X_train: np.ndarray, X_test: np.ndarray):
    """仅用训练集拟合输入特征标准化器，并同步变换测试集，这样可以避免测试集信息提前泄漏到训练阶段。"""
    n_features = X_train.shape[-1]
    scaler_X = StandardScaler()
    scaler_X.fit(X_train.reshape(-1, n_features))
    X_train_scaled = scaler_X.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test_scaled = scaler_X.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)
    return X_train_scaled, X_test_scaled, scaler_X


def fit_transform_Y_multi(Y_train: np.ndarray, Y_test: np.ndarray):
    """仅用训练集拟合多步标签标准化器，并同步变换测试集，这样可以避免测试集信息提前泄漏到训练阶段。"""
    scaler_Y = StandardScaler()
    scaler_Y.fit(Y_train)
    Y_train_scaled = scaler_Y.transform(Y_train)
    Y_test_scaled = scaler_Y.transform(Y_test)
    return Y_train_scaled, Y_test_scaled, scaler_Y


def evaluate_direct_multi(model: nn.Module,
                          X_test: np.ndarray,
                          Y_test_scaled: np.ndarray,
                          Y_test_raw: np.ndarray,
                          scaler_Y: StandardScaler,
                          batch_size: int):
    """对直接多步预测模型做反标准化和指标评估，方便在阅读主流程时快速理解它的职责。"""
    # 先把测试集封装成 DataLoader，便于按批次完成推理。
    loader = create_test_loader(X_test, Y_test_scaled, batch_size=batch_size)
    criterion = nn.HuberLoss(delta=1.0)

    # 在不更新参数的情况下完成测试集预测。
    _, preds_scaled, _ = run_epoch(model, loader, criterion, optimizer=None)
    # 把预测值从标准化空间还原回真实价格尺度。
    preds_raw = scaler_Y.inverse_transform(preds_scaled)

    horizon_metrics = {}
    for h in range(FORECAST_HORIZONS):
        horizon_metrics[f"h{h+1}"] = regression_metrics(Y_test_raw[:, h], preds_raw[:, h])

    # 同时统计整体指标与分步指标，方便从不同角度评价模型效果。
    overall_metrics = regression_metrics(Y_test_raw.reshape(-1), preds_raw.reshape(-1))
    return preds_raw, horizon_metrics, overall_metrics


# ============================================================
# 绘图函数
# ============================================================

def plot_training_curve(history: Dict[str, List[float]],
                        title: str,
                        save_path: Path):
    """绘制训练集和验证集的损失曲线，图像结果会被保存下来，方便后续分析和论文展示。"""
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="训练集损失")
    plt.plot(history["val_loss"], label="验证集损失")
    plt.title(title)
    plt.xlabel("训练轮次")
    plt.ylabel("Huber损失")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.close()


def plot_three_horizon_prediction(target_dates: np.ndarray,
                                  y_true: np.ndarray,
                                  y_pred: np.ndarray,
                                  title_prefix: str,
                                  save_path: Path):
    """绘制三个预测步长下的真实值与预测值对比图，图像结果会被保存下来，方便后续分析和论文展示。"""
    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)

    for h in range(FORECAST_HORIZONS):
        dt = pd.to_datetime(target_dates[:, h])
        axes[h].plot(dt, y_true[:, h], label="真实值", linewidth=1.8)
        axes[h].plot(dt, y_pred[:, h], label="预测值", linewidth=1.6)
        axes[h].set_title(f"{title_prefix} - 第 t+{h+1} 天预测")
        axes[h].set_ylabel("收盘价")
        axes[h].grid(alpha=0.3)
        axes[h].legend()

    axes[-1].set_xlabel("日期")
    plt.tight_layout()
    plt.savefig(save_path, dpi=180)
    plt.close()


def plot_paper_arch_comparison(stock_dir: Path,
                               stock_label: str):
    """绘制论文展示用的两种基础结构对比图，图像结果会被保存下来，方便后续分析和论文展示。"""
    base_file = stock_dir / "direct_multi_BaseLSTM_predictions.csv"
    feat_file = stock_dir / "direct_multi_FeatureAttention_LSTM_predictions.csv"

    if (not base_file.exists()) or (not feat_file.exists()):
        return

    base_df = pd.read_csv(base_file)
    feat_df = pd.read_csv(feat_file)

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)

    for h in range(1, 4):
        date_col = f"date_t{h}"
        true_col = f"true_t{h}"
        pred_col = f"pred_t{h}"

        dt = pd.to_datetime(base_df[date_col])
        axes[h - 1].plot(dt, base_df[true_col], label="真实值", linewidth=1.9)
        axes[h - 1].plot(dt, base_df[pred_col], label="BaseLSTM", linewidth=1.5)
        axes[h - 1].plot(dt, feat_df[pred_col], label="FeatureAttention_LSTM", linewidth=1.5)
        axes[h - 1].set_title(f"{stock_label} - direct_multi - 第 t+{h} 天预测对比")
        axes[h - 1].grid(alpha=0.3)
        axes[h - 1].legend()

    axes[-1].set_xlabel("日期")
    plt.tight_layout()
    plt.savefig(stock_dir / "direct_multi_paper_arch_comparison.png", dpi=180)
    plt.close()


def plot_summary_bars(results_df: pd.DataFrame, save_dir: Path):
    """根据汇总结果绘制柱状图和折线图，图像结果会被保存下来，方便后续分析和论文展示。"""
    # 先按模型汇总整体指标和各预测步长的平均表现。
    summary = results_df.groupby(["arch_model"]).agg(
        mean_overall_R2=("overall_R2", "mean"),
        mean_overall_RMSE=("overall_RMSE", "mean"),
        mean_h1_R2=("h1_R2", "mean"),
        mean_h2_R2=("h2_R2", "mean"),
        mean_h3_R2=("h3_R2", "mean"),
    ).reset_index()

    labels = summary["arch_model"]

    # 先绘制整体 R2 的柱状图，比较不同模型的总体效果。
    plt.figure(figsize=(8, 5))
    plt.bar(labels, summary["mean_overall_R2"])
    plt.title("不同模型的平均整体R²")
    plt.ylabel("平均整体R²")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / "mean_overall_r2_bar.png", dpi=180)
    plt.close()

    plt.figure(figsize=(8, 5))
    # 再绘制整体 RMSE 柱状图，从误差角度补充比较结果。
    plt.bar(labels, summary["mean_overall_RMSE"])
    plt.title("不同模型的平均整体RMSE")
    plt.ylabel("平均整体RMSE")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / "mean_overall_rmse_bar.png", dpi=180)
    plt.close()

    plt.figure(figsize=(8, 5))
    # 最后绘制不同预测步长下的平均 R2 曲线，观察性能随步长变化的趋势。
    x = np.array([1, 2, 3])
    for _, row in summary.iterrows():
        y = [row["mean_h1_R2"], row["mean_h2_R2"], row["mean_h3_R2"]]
        label = row["arch_model"]
        plt.plot(x, y, marker="o", label=label)
    plt.xticks([1, 2, 3], ["t+1", "t+2", "t+3"])
    plt.title("不同预测步长下的平均R²")
    plt.ylabel("R²")
    plt.xlabel("预测步长")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_dir / "mean_horizon_r2_line.png", dpi=180)
    plt.close()


def plot_r2_heatmap(results_df: pd.DataFrame, save_dir: Path):
    """绘制不同股票与模型的整体 R2 热力图，图像结果会被保存下来，方便后续分析和论文展示。"""
    pivot_df = results_df.pivot_table(
        index="name",
        columns="arch_model",
        values="overall_R2"
    )

    if pivot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(8, max(8, len(pivot_df) * 0.35)))
    im = ax.imshow(pivot_df.values, aspect="auto")

    ax.set_xticks(np.arange(len(pivot_df.columns)))
    ax.set_xticklabels(pivot_df.columns, rotation=20, ha="right", fontsize=9)

    ax.set_yticks(np.arange(len(pivot_df.index)))
    ax.set_yticklabels(pivot_df.index)

    ax.set_title("不同股票与模型的整体R²热力图")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("整体R²")

    for i in range(pivot_df.shape[0]):
        for j in range(pivot_df.shape[1]):
            val = pivot_df.iloc[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)

    plt.tight_layout()
    plt.savefig(save_dir / "overall_r2_heatmap.png", dpi=180)
    plt.close()


# ============================================================
# 结果整理
# ============================================================

def prediction_matrix_to_df(target_dates: np.ndarray,
                            y_true: np.ndarray,
                            y_pred: np.ndarray) -> pd.DataFrame:
    """把多步预测矩阵整理成便于保存的表格，方便在阅读主流程时快速理解它的职责。"""
    return pd.DataFrame({
        "date_t1": pd.to_datetime(target_dates[:, 0]),
        "true_t1": y_true[:, 0],
        "pred_t1": y_pred[:, 0],
        "date_t2": pd.to_datetime(target_dates[:, 1]),
        "true_t2": y_true[:, 1],
        "pred_t2": y_pred[:, 1],
        "date_t3": pd.to_datetime(target_dates[:, 2]),
        "true_t3": y_true[:, 2],
        "pred_t3": y_pred[:, 2],
    })


def metrics_row(symbol: str,
                name: str,
                sector: str,
                arch_model: str,
                params: int,
                n_samples: int,
                n_train: int,
                n_test: int,
                horizon_metrics: Dict[str, Dict[str, float]],
                overall_metrics: Dict[str, float],
                best_config: Dict[str, Any]):
    """把单个模型的评估结果整理成一行汇总记录，方便在阅读主流程时快速理解它的职责。"""
    # 把当前模型的基础信息、样本规模和评估指标整理成一条标准记录。
    row = {
        "symbol": symbol,
        "name": name,
        "sector": sector,
        "experiment_mode": "direct_multi",
        "arch_model": arch_model,
        "params": params,
        "samples_total": n_samples,
        "train_samples": n_train,
        "test_samples": n_test,
        "overall_MSE": overall_metrics["MSE"],
        "overall_RMSE": overall_metrics["RMSE"],
        "overall_MAE": overall_metrics["MAE"],
        "overall_R2": overall_metrics["R2"],
        "h1_MSE": horizon_metrics["h1"]["MSE"],
        "h1_RMSE": horizon_metrics["h1"]["RMSE"],
        "h1_MAE": horizon_metrics["h1"]["MAE"],
        "h1_R2": horizon_metrics["h1"]["R2"],
        "h2_MSE": horizon_metrics["h2"]["MSE"],
        "h2_RMSE": horizon_metrics["h2"]["RMSE"],
        "h2_MAE": horizon_metrics["h2"]["MAE"],
        "h2_R2": horizon_metrics["h2"]["R2"],
        "h3_MSE": horizon_metrics["h3"]["MSE"],
        "h3_RMSE": horizon_metrics["h3"]["RMSE"],
        "h3_MAE": horizon_metrics["h3"]["MAE"],
        "h3_R2": horizon_metrics["h3"]["R2"],
        "best_config": json.dumps(best_config, ensure_ascii=False),
    }
    return row


# ============================================================
# 单股实验
# ============================================================

def run_single_stock_experiment(stock_info: Dict[str, str],
                                start_date: str = START_DATE,
                                end_date: str = END_DATE,
                                lookback: int = LOOKBACK,
                                verbose: bool = True):
    """完成单只股票从数据抓取、特征构造、样本切分、模型训练到结果保存的完整实验流程，便于整体把握单股实验是怎样一步步推进的。"""
    # 先解析当前股票的代码、名称和所属板块等基础信息。
    symbol = stock_info["symbol"]
    name = stock_info["name"]
    sector = stock_info["sector"]
    stock_label = f"{symbol}_{name}"

    # 为当前股票创建独立输出目录，后续结果都会存到这里。
    stock_dir = ROOT_DIR / stock_label
    safe_mkdir(stock_dir)

    print("\n" + "#" * 96)
    print(f"处理股票: {stock_label} | 板块={sector}")
    print("#" * 96)

    # 先抓取原始行情数据，再构造训练所需的特征表。
    raw_df = fetch_single_stock_data(symbol, start_date, end_date)
    feature_df = create_features(raw_df)

    # 如果特征工程后的样本太少，就直接跳过这只股票。
    if len(feature_df) < lookback + FORECAST_HORIZONS + 30:
        raise ValueError(f"{stock_label}: 特征工程后有效行数过少 ({len(feature_df)})")

    # 把时间序列切成监督学习样本，并保留未来日期信息。
    X_all, Y_all, target_dates_all, feature_cols = build_multi_horizon_dataset(
        feature_df, lookback=lookback, horizons=FORECAST_HORIZONS
    )

    n_samples = len(X_all)
    train_idx, test_idx = chronological_split_indices(n_samples, TEST_RATIO)

    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
    Y_train_raw, Y_test_raw = Y_all[train_idx], Y_all[test_idx]
    dates_test = target_dates_all[test_idx]

    # 只用训练集拟合标准化器，再同步变换训练集和测试集。
    X_train_scaled, X_test_scaled, scaler_X = fit_transform_X(X_train_raw, X_test_raw)
    Y_train_scaled, Y_test_scaled, scaler_Y = fit_transform_Y_multi(Y_train_raw, Y_test_raw)

    # 保存标准化器和特征列，方便后续复现实验或单独推理。
    joblib.dump(scaler_X, stock_dir / "scaler_X.pkl")
    joblib.dump(scaler_Y, stock_dir / "scaler_Y.pkl")
    pd.Series(feature_cols, name="feature").to_csv(stock_dir / "feature_columns.csv", index=False)

    input_dim = X_train_scaled.shape[-1]
    all_rows = []

    # 依次训练每个候选模型，并分别记录它们的最佳表现。
    for arch in ARCH_NAMES:
        print(f"\n[{stock_label}] direct_multi | {arch}")

        # 先对当前模型调参，再用最优配置恢复模型权重。
        best_result = tune_model(
            model_name=arch,
            X_train=X_train_scaled,
            y_train=Y_train_scaled,
            input_dim=input_dim,
            out_dim=FORECAST_HORIZONS,
            verbose=verbose
        )

        best_cfg = best_result["config"]
        model = build_model(arch, input_dim, best_cfg, out_dim=FORECAST_HORIZONS)
        model.load_state_dict(best_result["state_dict"])

        # 使用最佳模型在测试集上预测，并计算多步评估指标。
        preds_raw, horizon_metrics, overall_metrics = evaluate_direct_multi(
            model=model,
            X_test=X_test_scaled,
            Y_test_scaled=Y_test_scaled,
            Y_test_raw=Y_test_raw,
            scaler_Y=scaler_Y,
            batch_size=best_cfg.batch_size
        )

        params = count_parameters(model)

        # 把预测结果、模型权重和训练可视化图按文件保存下来。
        prediction_matrix_to_df(dates_test, Y_test_raw, preds_raw).to_csv(
            stock_dir / f"direct_multi_{arch}_predictions.csv", index=False
        )

        torch.save(model.state_dict(), stock_dir / f"direct_multi_{arch}_best_model.pth")

        plot_training_curve(
            best_result["history"],
            f"{stock_label} - direct_multi - {arch} 训练过程",
            stock_dir / f"direct_multi_{arch}_training_curve.png"
        )

        plot_three_horizon_prediction(
            dates_test, Y_test_raw, preds_raw,
            f"{stock_label} - direct_multi - {arch}",
            stock_dir / f"direct_multi_{arch}_prediction.png"
        )

        # 将当前模型的指标整理成一行，后续用于该股票内部汇总。
        all_rows.append(metrics_row(
            symbol, name, sector, arch,
            params, n_samples, len(train_idx), len(test_idx),
            horizon_metrics, overall_metrics, asdict(best_cfg)
        ))

    # 当前股票所有模型跑完后，再额外生成论文展示用对比图。
    plot_paper_arch_comparison(stock_dir, stock_label)

    stock_result_df = pd.DataFrame(all_rows)
    stock_result_df.to_csv(stock_dir / "stock_all_results.csv", index=False)
    return stock_result_df


# ============================================================
# 全局汇总
# ============================================================

def summarize_all_results(all_result_df: pd.DataFrame):
    """汇总所有股票的实验结果并生成全局图表，方便从整体上比较不同模型或不同股票的表现。"""
    # 先保存所有股票拼接后的原始结果表，保留完整实验记录。
    all_result_df.to_csv(ROOT_DIR / "all_stock_direct_multi_results.csv", index=False)

    # 再按模型汇总平均 R2、RMSE 和参数规模等全局指标。
    summary = all_result_df.groupby(["arch_model"]).agg(
        stocks=("symbol", "count"),
        mean_overall_R2=("overall_R2", "mean"),
        std_overall_R2=("overall_R2", "std"),
        mean_overall_RMSE=("overall_RMSE", "mean"),
        std_overall_RMSE=("overall_RMSE", "std"),
        mean_h1_R2=("h1_R2", "mean"),
        mean_h2_R2=("h2_R2", "mean"),
        mean_h3_R2=("h3_R2", "mean"),
        mean_params=("params", "mean"),
    ).reset_index()

    # 按整体 R2 从高到低排序，便于直接看出模型优劣。
    summary = summary.sort_values("mean_overall_R2", ascending=False)
    summary.to_csv(ROOT_DIR / "summary_direct_multi.csv", index=False)

    # 输出全局柱状图和热力图，帮助从宏观角度比较不同模型。
    plot_summary_bars(all_result_df, ROOT_DIR)
    plot_r2_heatmap(all_result_df, ROOT_DIR)

    # 同时在控制台打印简洁汇总，方便快速查看实验结论。
    print("\n" + "=" * 130)
    print("直接多步预测总体汇总")
    print("=" * 130)
    print(f"{'arch_model':<26}{'mean_overall_R2':<18}{'mean_overall_RMSE':<20}{'mean_h1_R2':<14}{'mean_h2_R2':<14}{'mean_h3_R2':<14}")
    print("=" * 130)
    for _, row in summary.iterrows():
        print(
            f"{row['arch_model']:<26}"
            f"{row['mean_overall_R2']:<18.4f}"
            f"{row['mean_overall_RMSE']:<20.6f}"
            f"{row['mean_h1_R2']:<14.4f}"
            f"{row['mean_h2_R2']:<14.4f}"
            f"{row['mean_h3_R2']:<14.4f}"
        )
    print("=" * 130)


# ============================================================
# 主入口
# ============================================================

def main():
    """作为基础实验的主入口，按股票顺序执行全量实验，并在最后统一汇总所有成功结果。"""
    print("=" * 110)
    print("25支股票 PyTorch GPU 最终实验：仅使用 direct_multi")
    print("=" * 110)
    print(f"Using device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
        print(f"Initial allocated memory: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

    print(f"Stock count: {len(STOCK_POOL)}")
    print(f"Architectures: {ARCH_NAMES}")
    print(f"Lookback: {LOOKBACK}")
    print(f"Horizons: [t+1, t+2, t+3]")
    print(f"Date range: {START_DATE} -> {END_DATE}")

    # 先保存当前实验使用的股票池，确保实验对象可以追溯。
    pd.DataFrame(STOCK_POOL).to_csv(ROOT_DIR / "stock_pool.csv", index=False)

    all_result_list = []

    # 按股票顺序逐个执行实验，单只失败不会中断整个流程。
    for stock_info in STOCK_POOL:
        try:
            stock_df = run_single_stock_experiment(
                stock_info=stock_info,
                start_date=START_DATE,
                end_date=END_DATE,
                lookback=LOOKBACK,
                verbose=True
            )
            all_result_list.append(stock_df)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"[Error] Failed on stock {stock_info['symbol']} - {stock_info['name']}: {e}")

    # 如果所有股票都失败，则直接结束程序并给出提示。
    if not all_result_list:
        print("[Error] All stock experiments failed.")
        return

    # 只要有成功结果，就继续做全局汇总和图形化分析。
    all_result_df = pd.concat(all_result_list, ignore_index=True)
    summarize_all_results(all_result_df)

    # 最后检查论文展示用股票的对比图是否已经生成。
    paper_stock = next((s for s in STOCK_POOL if s["symbol"] == PAPER_STOCK_SYMBOL), None)
    if paper_stock is not None:
        stock_label = f"{paper_stock['symbol']}_{paper_stock['name']}"
        stock_dir = ROOT_DIR / stock_label
        fig_path = stock_dir / "direct_multi_paper_arch_comparison.png"
        if fig_path.exists():
            print(f"[Info] 论文对比图已保存: {fig_path}")

    print(f"\n全部结果已保存到: {ROOT_DIR.resolve()}")


if __name__ == "__main__":
    main()

25支股票 PyTorch GPU 最终实验：仅使用 direct_multi
Using device: cuda
GPU name: NVIDIA GeForce RTX 3070 Laptop GPU
Initial allocated memory: 0.00 MB
Stock count: 25
Architectures: ['BaseLSTM', 'FeatureAttention_LSTM']
Lookback: 90
Horizons: [t+1, t+2, t+3]
Date range: 20160101 -> 20250701

################################################################################################
处理股票: 600036_招商银行 | 板块=银行
################################################################################################

[600036_招商银行] direct_multi | BaseLSTM

调参 BaseLSTM

  试验 1/3 -> TrainConfig(hidden_dim=64, num_layers=1, dropout=0.2, learning_rate=0.001, batch_size=32, max_epochs=100, patience=12, weight_decay=0.0001)
    Epoch [001/100] train=0.379259 val=0.023429 lr=0.001
    Epoch [002/100] train=0.056734 val=0.028620 lr=0.001
    Epoch [003/100] train=0.055991 val=0.012280 lr=0.001
    Epoch [004/100] train=0.018700 val=0.009692 lr=0.001
    Epoch [005/100] train=0.021549 val=0.014912 lr=0.001
    Epoch 

In [5]:
# =========================
# 阶段一实现：10 只股票结构筛选
# 保持预测目标不变：预测 [t+1, t+2, t+3] 收盘价
# =========================

# -------- 配置 --------
SCREEN_STOCK_SYMBOLS = [
    "600036", "000001", "600519", "000858", "002594",
    "300750", "601633", "600276", "002415", "601899",
]

SCREEN_ARCH_NAMES = [
    "BaseLSTM",
    "FeatureAttention_LSTM",
    "FeatureAttention_TCN_LSTM",
    "FeatureAttention_Transformer_LSTM",
    "FeatureAttention_Graph_LSTM",
]

SCREEN_OUTER_DIR = ROOT_DIR / "screening_10stocks"
safe_mkdir(SCREEN_OUTER_DIR)

# 阶段一先使用较轻量的搜索预算做结构筛选
SCREEN_SEARCH_SPACE = [
    TrainConfig(hidden_dim=64, num_layers=1, dropout=0.2, learning_rate=1e-3, batch_size=32, max_epochs=50, patience=8),
    TrainConfig(hidden_dim=96, num_layers=1, dropout=0.2, learning_rate=5e-4, batch_size=32, max_epochs=60, patience=10),
]


# -------- 模型扩展 --------
class TemporalConvBlock(nn.Module):
    """在时间维上提取局部模式的卷积残差块，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(self, hidden_dim: int, kernel_size: int = 3):
        """初始化用于时间维特征提取的卷积残差块，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=kernel_size, padding=pad)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=kernel_size, padding=pad)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.act = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """在时间维做卷积变换，并保留残差连接，输入数据会在这里按照当前模块的设计完成主要计算。"""
        # x 的形状为 [B, T, H]
        z = x.transpose(1, 2)  # 转成 [B, H, T] 以适配一维卷积
        residual = z
        z = self.act(self.bn1(self.conv1(z)))
        z = self.bn2(self.conv2(z))
        z = self.act(z + residual)
        return z.transpose(1, 2)  # 再转回 [B, T, H]


class FeatureAttentionTCNLSTM(nn.Module):
    """特征注意力、LSTM 和 TCN 组合模型，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        out_dim: int = 3,
    ):
        """初始化特征注意力、LSTM 和 TCN 组合网络，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        self.feature_attn = FeatureAttention(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.tcn = TemporalConvBlock(hidden_dim=hidden_dim, kernel_size=3)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        """依次经过特征注意力、LSTM 和 TCN 后输出预测结果，输入数据会在这里按照当前模块的设计完成主要计算。"""
        z, feat_weights = self.feature_attn(x)
        seq, _ = self.lstm(z)
        seq = self.tcn(seq)
        last = seq[:, -1, :]
        pred = self.fc(self.dropout(last))
        return pred, feat_weights.mean(dim=1)


class FeatureAttentionTransformerLSTM(nn.Module):
    """特征注意力、LSTM 和 Transformer 组合模型，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        out_dim: int = 3,
        nhead: int = 4,
    ):
        """初始化特征注意力、LSTM 和 Transformer 组合网络，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        self.feature_attn = FeatureAttention(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        """依次经过特征注意力、LSTM 和 Transformer 后输出预测结果，输入数据会在这里按照当前模块的设计完成主要计算。"""
        z, feat_weights = self.feature_attn(x)
        seq, _ = self.lstm(z)
        seq = self.encoder(seq)
        seq = self.norm(seq)
        last = seq[:, -1, :]
        pred = self.fc(self.dropout(last))
        return pred, feat_weights.mean(dim=1)


class FeatureGraphLayer(nn.Module):
    """根据特征图邻接矩阵做一次特征传播，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(self, input_dim: int):
        """初始化特征图传播层中的可学习混合系数，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        self.input_dim = input_dim
        self.alpha = nn.Parameter(torch.tensor(0.5, dtype=torch.float32))

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """结合邻接矩阵对特征做一次图传播，输入数据会在这里按照当前模块的设计完成主要计算。"""
        # x 的形状为 [B, T, F]，adj 的形状为 [F, F]
        # 在特征图上做一次简单的信息传播。
        support = torch.matmul(x, adj)
        mix = torch.sigmoid(self.alpha)
        return mix * support + (1.0 - mix) * x


class FeatureAttentionGraphLSTM(nn.Module):
    """特征图传播、特征注意力和 LSTM 组合模型，读者可以据此更快理解这个类在整个实验流程中承担的角色。"""
    def __init__(
        self,
        input_dim: int,
        graph_adj: torch.Tensor,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        out_dim: int = 3,
    ):
        """初始化特征图、注意力和 LSTM 组合网络，这里会提前准备后续计算所需的层、参数或缓存对象。"""
        super().__init__()
        self.register_buffer("graph_adj", graph_adj)
        self.graph_layer = FeatureGraphLayer(input_dim=input_dim)
        self.feature_attn = FeatureAttention(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        """先做特征图传播，再做注意力和 LSTM 预测，输入数据会在这里按照当前模块的设计完成主要计算。"""
        gx = self.graph_layer(x, self.graph_adj)
        z, feat_weights = self.feature_attn(gx)
        seq, _ = self.lstm(z)
        last = seq[:, -1, :]
        pred = self.fc(self.dropout(last))
        return pred, feat_weights.mean(dim=1)


# -------- 筛选辅助函数 --------
def get_screen_stock_pool(stock_pool: List[Dict[str, str]], symbols: List[str]) -> List[Dict[str, str]]:
    """从总股票池中筛出阶段一要使用的股票列表，这样主流程可以更统一地获取所需对象或结果。"""
    symbol_set = set(symbols)
    selected = [s for s in stock_pool if s["symbol"] in symbol_set]
    if len(selected) != len(symbol_set):
        found = {s["symbol"] for s in selected}
        missing = sorted(symbol_set - found)
        raise ValueError(f"screening symbols missing in STOCK_POOL: {missing}")
    return selected


def build_feature_graph_from_X(X_train: np.ndarray, top_k: int = 6) -> np.ndarray:
    """根据训练样本中的特征相关性构造特征图邻接矩阵，为图结构模型提供可传播的特征关系信息。"""
    # X_train 的形状为 [N, T, F]
    # 先把样本展平到特征维度上，方便统计特征之间的相关性。
    n_features = X_train.shape[-1]
    flat = X_train.reshape(-1, n_features)
    # 根据训练数据计算特征相关系数矩阵，作为构图依据。
    corr = np.corrcoef(flat, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    corr = np.abs(corr)

    # 只保留每个特征最相关的前 k 个邻居，减少噪声连接。
    # 只保留最相关的若干邻居，避免特征图连接过于稠密。
    adj = np.zeros_like(corr, dtype=np.float32)
    for i in range(n_features):
        idx = np.argsort(corr[i])[-(top_k + 1):]
        adj[i, idx] = corr[i, idx]
    adj = np.maximum(adj, adj.T)
    np.fill_diagonal(adj, 1.0)

    # 最后对邻接矩阵做归一化，让后续图传播更稳定。
    row_sum = adj.sum(axis=1, keepdims=True) + 1e-8
    adj = adj / row_sum
    return adj.astype(np.float32)


def build_screen_model(
    model_name: str,
    input_dim: int,
    cfg: TrainConfig,
    out_dim: int,
    graph_adj_np: np.ndarray = None,
) -> nn.Module:
    """根据名称构造阶段一筛选使用的候选模型，这样后续主流程就可以通过统一接口直接复用。"""
    if model_name == "BaseLSTM":
        return BaseLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "FeatureAttention_LSTM":
        return FeatureAttentionLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "FeatureAttention_TCN_LSTM":
        return FeatureAttentionTCNLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "FeatureAttention_Transformer_LSTM":
        return FeatureAttentionTransformerLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "FeatureAttention_Graph_LSTM":
        if graph_adj_np is None:
            raise ValueError("graph_adj_np is required for FeatureAttention_Graph_LSTM")
        graph_adj = torch.tensor(graph_adj_np, dtype=torch.float32, device=DEVICE)
        return FeatureAttentionGraphLSTM(
            input_dim=input_dim,
            graph_adj=graph_adj,
            hidden_dim=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            dropout=cfg.dropout,
            out_dim=out_dim,
        ).to(DEVICE)
    raise ValueError(f"Unknown screening model: {model_name}")


def tune_screen_model(
    model_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    input_dim: int,
    out_dim: int,
    graph_adj_np: np.ndarray = None,
    verbose: bool = True,
):
    """使用阶段一的轻量搜索空间为候选模型调参，重点是在较低计算成本下快速比较结构优劣。"""
    # 先准备一个变量，用来记录筛选阶段当前最优的配置结果。
    best_result = None

    if verbose:
        print("\n" + "=" * 72)
        print(f"Screening tune -> {model_name}")
        print("=" * 72)

    # 逐个轻量配置进行训练，控制阶段一的计算成本。
    for i, cfg in enumerate(SCREEN_SEARCH_SPACE, 1):
        if verbose:
            print(f"\n  Config {i}/{len(SCREEN_SEARCH_SPACE)} -> {cfg}")

        # 构造当前候选模型，并准备对应的训练和验证数据。
        model = build_screen_model(
            model_name=model_name,
            input_dim=input_dim,
            cfg=cfg,
            out_dim=out_dim,
            graph_adj_np=graph_adj_np,
        )

        train_loader, val_loader = create_train_val_loaders(
            X_train,
            y_train,
            batch_size=cfg.batch_size,
            val_ratio=VAL_RATIO_WITHIN_TRAIN,
        )

        # 运行训练流程，记录该配置在验证集上的最佳表现。
        model, history, best_val_loss = train_with_early_stopping(
            model,
            train_loader,
            val_loader,
            cfg,
            verbose=verbose,
        )

        result = {
            "config": cfg,
            "history": history,
            "best_val_loss": best_val_loss,
            "state_dict": copy.deepcopy(model.state_dict()),
        }

        # 持续比较各配置结果，只保留当前表现最好的那一组。
        if best_result is None or best_val_loss < best_result["best_val_loss"]:
            best_result = result

    return best_result


def run_single_stock_screening(stock_info: Dict[str, str], verbose: bool = True) -> pd.DataFrame:
    """对单只股票执行阶段一结构筛选实验，会串起数据准备、轻量调参、测试评估和结果保存等关键步骤。"""
    # 先拆出当前股票的基础信息，后续日志和文件命名都要用到。
    symbol = stock_info["symbol"]
    name = stock_info["name"]
    sector = stock_info["sector"]
    stock_label = f"{symbol}_{name}"

    # 为当前筛选股票创建独立目录，便于保存各类中间结果。
    stock_dir = SCREEN_OUTER_DIR / stock_label
    safe_mkdir(stock_dir)

    print("\n" + "#" * 96)
    print(f"[SCREEN] 处理股票: {stock_label} | 板块={sector}")
    print("#" * 96)

    # 先抓取原始行情数据，再构造阶段一需要的特征和样本。
    raw_df = fetch_single_stock_data(symbol, START_DATE, END_DATE)
    feature_df = create_features(raw_df)

    # 若可用样本太少，则直接跳过，避免后续训练不稳定。
    if len(feature_df) < LOOKBACK + FORECAST_HORIZONS + 30:
        raise ValueError(f"{stock_label}: valid rows too few after feature engineering ({len(feature_df)})")

    # 把时间序列切成监督学习样本，并准备按时间顺序划分训练与测试。
    X_all, Y_all, target_dates_all, feature_cols = build_multi_horizon_dataset(
        feature_df,
        lookback=LOOKBACK,
        horizons=FORECAST_HORIZONS,
    )

    n_samples = len(X_all)
    train_idx, test_idx = chronological_split_indices(n_samples, TEST_RATIO)

    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
    Y_train_raw, Y_test_raw = Y_all[train_idx], Y_all[test_idx]
    dates_test = target_dates_all[test_idx]

    # 对输入和标签分别标准化，同时保证只在训练集上拟合。
    X_train_scaled, X_test_scaled, scaler_X = fit_transform_X(X_train_raw, X_test_raw)
    Y_train_scaled, Y_test_scaled, scaler_Y = fit_transform_Y_multi(Y_train_raw, Y_test_raw)

    # 根据训练数据构造特征图，为图结构模型提供邻接矩阵。
    graph_adj_np = build_feature_graph_from_X(X_train_scaled, top_k=6)

    # 保存标准化器和特征列，方便后续复用或排查问题。
    joblib.dump(scaler_X, stock_dir / "scaler_X.pkl")
    joblib.dump(scaler_Y, stock_dir / "scaler_Y.pkl")
    pd.Series(feature_cols, name="feature").to_csv(stock_dir / "feature_columns.csv", index=False)

    input_dim = X_train_scaled.shape[-1]
    all_rows = []

    # 依次训练阶段一的各个候选模型，并记录筛选结果。
    for arch in SCREEN_ARCH_NAMES:
        print(f"\n[{stock_label}] screening | {arch}")

        # 先用轻量搜索空间调参，再恢复该模型的最佳权重。
        best_result = tune_screen_model(
            model_name=arch,
            X_train=X_train_scaled,
            y_train=Y_train_scaled,
            input_dim=input_dim,
            out_dim=FORECAST_HORIZONS,
            graph_adj_np=graph_adj_np,
            verbose=verbose,
        )

        best_cfg = best_result["config"]
        model = build_screen_model(
            model_name=arch,
            input_dim=input_dim,
            cfg=best_cfg,
            out_dim=FORECAST_HORIZONS,
            graph_adj_np=graph_adj_np,
        )
        model.load_state_dict(best_result["state_dict"])

        # 在测试集上生成预测并计算指标，用于比较结构优劣。
        preds_raw, horizon_metrics, overall_metrics = evaluate_direct_multi(
            model=model,
            X_test=X_test_scaled,
            Y_test_scaled=Y_test_scaled,
            Y_test_raw=Y_test_raw,
            scaler_Y=scaler_Y,
            batch_size=best_cfg.batch_size,
        )

        params = count_parameters(model)

        # 把预测结果、模型权重和训练曲线文件按股票分别保存。
        prediction_matrix_to_df(dates_test, Y_test_raw, preds_raw).to_csv(
            stock_dir / f"direct_multi_{arch}_predictions.csv", index=False
        )

        torch.save(model.state_dict(), stock_dir / f"direct_multi_{arch}_best_model.pth")

        plot_training_curve(
            best_result["history"],
            f"{stock_label} - screening - {arch} training",
            stock_dir / f"direct_multi_{arch}_training_curve.png",
        )
        plot_three_horizon_prediction(
            dates_test,
            Y_test_raw,
            preds_raw,
            f"{stock_label} - screening - {arch}",
            stock_dir / f"direct_multi_{arch}_prediction.png",
        )

        # 将当前模型的评估结果整理进该股票的汇总表。
        all_rows.append(
            metrics_row(
                symbol,
                name,
                sector,
                arch,
                params,
                n_samples,
                len(train_idx),
                len(test_idx),
                horizon_metrics,
                overall_metrics,
                asdict(best_cfg),
            )
        )

    stock_result_df = pd.DataFrame(all_rows)
    stock_result_df.to_csv(stock_dir / "stock_all_results.csv", index=False)
    return stock_result_df


def summarize_screening_results(all_result_df: pd.DataFrame, save_dir: Path):
    """汇总阶段一结果，并计算相对注意力基线的对比指标，方便判断哪些结构值得保留到后续全量实验。"""
    # 先确保汇总目录存在，并保存阶段一的完整原始结果。
    safe_mkdir(save_dir)
    all_result_df.to_csv(save_dir / "all_10stocks_screening_results.csv", index=False)

    # 按模型统计平均 R2、MAE 以及参数规模等关键指标。
    summary = all_result_df.groupby(["arch_model"]).agg(
        stocks=("symbol", "count"),
        mean_overall_R2=("overall_R2", "mean"),
        std_overall_R2=("overall_R2", "std"),
        mean_overall_MAE=("overall_MAE", "mean"),
        std_overall_MAE=("overall_MAE", "std"),
        mean_h1_R2=("h1_R2", "mean"),
        mean_h2_R2=("h2_R2", "mean"),
        mean_h3_R2=("h3_R2", "mean"),
        mean_params=("params", "mean"),
    ).reset_index()

    # 以特征注意力模型为基线，计算其它结构的增减幅度。
    baseline = summary.loc[summary["arch_model"] == "FeatureAttention_LSTM"]
    if len(baseline) == 1:
        base_r2 = float(baseline.iloc[0]["mean_overall_R2"])
        base_mae = float(baseline.iloc[0]["mean_overall_MAE"])
        summary["delta_vs_featattn_r2"] = summary["mean_overall_R2"] - base_r2
        summary["delta_vs_featattn_mae"] = summary["mean_overall_MAE"] - base_mae
    else:
        summary["delta_vs_featattn_r2"] = np.nan
        summary["delta_vs_featattn_mae"] = np.nan

    # 按整体 R2 排序，便于直接看出筛选阶段更优的结构。
    summary = summary.sort_values("mean_overall_R2", ascending=False)
    summary.to_csv(save_dir / "summary_10stocks_screening.csv", index=False)

    # 按股票统计相对 FeatureAttention_LSTM 的胜率
    # 再从单股维度统计各模型相对基线的胜率情况。
    stock_model_r2 = all_result_df.pivot_table(index="symbol", columns="arch_model", values="overall_R2")
    win_rows = []
    if "FeatureAttention_LSTM" in stock_model_r2.columns:
        base = stock_model_r2["FeatureAttention_LSTM"]
        for col in stock_model_r2.columns:
            if col == "FeatureAttention_LSTM":
                continue
            wins = (stock_model_r2[col] > base).sum()
            valid = (~stock_model_r2[col].isna() & ~base.isna()).sum()
            win_rows.append({
                "arch_model": col,
                "win_count": int(wins),
                "valid_count": int(valid),
                "win_rate": float(wins / valid) if valid > 0 else np.nan,
            })

    # 把胜率结果整理成表，如果非空就单独保存。
    win_df = pd.DataFrame(win_rows)
    if not win_df.empty:
        win_df.to_csv(save_dir / "win_rate_vs_feature_attention.csv", index=False)

    # 输出图形化汇总，帮助直观看到结构差异和总体趋势。
    plot_summary_bars(all_result_df, save_dir)
    plot_r2_heatmap(all_result_df, save_dir)

    # 最后在控制台打印一份简洁摘要，便于快速查看筛选结论。
    print("\n" + "=" * 132)
    print("10股筛选汇总 (LSTM主干 + Attention + X)")
    print("=" * 132)
    print(f"{'arch_model':<38}{'mean_overall_R2':<18}{'mean_overall_MAE':<18}{'delta_vs_featattn_r2':<22}")
    print("=" * 132)
    for _, row in summary.iterrows():
        print(
            f"{row['arch_model']:<38}"
            f"{row['mean_overall_R2']:<18.4f}"
            f"{row['mean_overall_MAE']:<18.4f}"
            f"{row['delta_vs_featattn_r2']:<22.4f}"
        )
    print("=" * 132)


# -------- 阶段一执行入口 --------
def run_stage1_screening_10stocks(verbose: bool = True):
    """执行 10 只股票的阶段一结构筛选入口，用较小计算预算先判断候选结构是否有继续扩展的价值。"""
    print("=" * 120)
    print("Stage-1: 10-stock architecture screening (keep LSTM + 3-day close prediction)")
    print("=" * 120)

    # 先从总股票池中筛出阶段一要跑的 10 只代表股票。
    screen_pool = get_screen_stock_pool(STOCK_POOL, SCREEN_STOCK_SYMBOLS)
    print(f"Screen stock count: {len(screen_pool)}")
    print(f"Screen architectures: {SCREEN_ARCH_NAMES}")

    # 把阶段一股票池保存下来，便于复现实验范围。
    pd.DataFrame(screen_pool).to_csv(SCREEN_OUTER_DIR / "screen_stock_pool.csv", index=False)

    all_result_list = []
    # 逐只股票执行结构筛选，单只失败不会阻塞整体流程。
    for stock_info in screen_pool:
        try:
            stock_df = run_single_stock_screening(stock_info=stock_info, verbose=verbose)
            all_result_list.append(stock_df)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"[Error] Stage-1 failed on {stock_info['symbol']} - {stock_info['name']}: {e}")

    # 如果全部股票都失败，则直接结束本阶段并给出提示。
    if not all_result_list:
        print("[Error] Stage-1 failed on all screening stocks.")
        return

    # 对所有成功结果做统一汇总，并生成阶段一总表。
    all_result_df = pd.concat(all_result_list, ignore_index=True)
    summarize_screening_results(all_result_df, SCREEN_OUTER_DIR)
    print(f"\nStage-1 results saved to: {SCREEN_OUTER_DIR.resolve()}")


print("Implementation ready. Run: run_stage1_screening_10stocks(verbose=True)")

Implementation ready. Run: run_stage1_screening_10stocks(verbose=True)


In [6]:
# =========================
# 补丁：新浪优先的数据接口覆盖
# 插入在筛选执行单元之前
# =========================

import time


def standardize_stock_df(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """把不同来源的股票数据统一成项目内部字段格式，方便在阅读主流程时快速理解它的职责。"""
    # 先把不同来源可能出现的列名统一映射到内部标准字段。
    df = df.rename(columns={
        "日期": "trade_date",
        "date": "trade_date",
        "开盘": "open",
        "open": "open",
        "最高": "high",
        "high": "high",
        "最低": "low",
        "low": "low",
        "收盘": "close",
        "close": "close",
        "成交量": "volume",
        "volume": "volume",
        "成交额": "amount",
        "amount": "amount",
        "换手率": "turnover",
        "turnover": "turnover",
    })

    # 检查关键行情字段是否齐全，缺失时直接报错。
    required = ["trade_date", "open", "high", "low", "close", "volume", "amount"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{symbol} stock data missing required columns: {missing}")

    if "turnover" not in df.columns:
        df["turnover"] = np.nan

    # 只保留必要字段，并统一日期与数值类型。
    keep_cols = ["trade_date", "open", "high", "low", "close", "volume", "amount", "turnover"]
    out = df[keep_cols].copy()
    out["trade_date"] = pd.to_datetime(out["trade_date"])

    for col in ["open", "high", "low", "close", "volume", "amount", "turnover"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # 最后按交易日排序并清理缺失值，输出规整后的数据表。
    out = out.sort_values("trade_date").dropna(subset=["trade_date", "close"]).reset_index(drop=True)
    return out



def to_market_prefixed_symbol(symbol: str) -> str:
    """把 6 位股票代码转换成带市场前缀的格式，方便在阅读主流程时快速理解它的职责。"""
    if symbol.startswith(("600", "601", "603", "605", "688", "689")):
        return f"sh{symbol}"
    return f"sz{symbol}"



def with_retry(func, max_retries: int = 4, sleep_min: float = 1.0, sleep_max: float = 2.5, desc: str = "request"):
    """为请求操作提供重试机制，降低偶发失败的影响，方便在阅读主流程时快速理解它的职责。"""
    # 先准备一个变量，用来记录多次重试中的最后一次异常。
    last_err = None
    # 按照设定的次数循环尝试执行请求操作。
    for i in range(1, max_retries + 1):
        try:
            return func()
        # 如果当前尝试失败，就打印日志并在必要时等待后重试。
        except Exception as e:
            last_err = e
            print(f"    [Retry-{i}/{max_retries}] {desc} failed: {repr(e)}")
            if i < max_retries:
                wait_sec = random.uniform(sleep_min, sleep_max)
                print(f"    [Retry-{i}/{max_retries}] retry after {wait_sec:.2f}s")
                time.sleep(wait_sec)
    raise RuntimeError(f"{desc} failed after {max_retries} retries. Last error: {repr(last_err)}")



def _try_stock_daily_sina(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """尝试通过新浪接口抓取指定股票的日线数据，方便在阅读主流程时快速理解它的职责。"""
    # 先把股票代码转换成新浪接口需要的市场前缀格式。
    market_symbol = to_market_prefixed_symbol(symbol)
    # 调用新浪接口抓取指定股票在目标区间内的日线数据。
    df = ak.stock_zh_a_daily(
        symbol=market_symbol,
        start_date=start_date,
        end_date=end_date,
        adjust=""
    )

    if df is None or len(df) == 0:
        raise RuntimeError(f"stock_zh_a_daily returned empty result: {symbol}")

    # 若日期在索引里，则先把它恢复成普通列，方便统一处理。
    if isinstance(df.index, pd.DatetimeIndex) and "date" not in df.columns:
        df = df.reset_index().rename(columns={df.index.name or "index": "date"})

    # 把接口返回结果整理成项目内部统一字段，并按日期范围裁剪。
    df = standardize_stock_df(df, symbol)
    df = df[
        (df["trade_date"] >= pd.to_datetime(start_date)) &
        (df["trade_date"] <= pd.to_datetime(end_date))
    ].copy().reset_index(drop=True)
    return df



def _try_stock_hist_em(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """尝试通过东方财富接口抓取指定股票的历史数据，方便在阅读主流程时快速理解它的职责。"""
    # 当新浪接口不可用时，改用东方财富接口作为兜底来源。
    df = ak.stock_zh_a_hist(
        symbol=symbol,
        period="daily",
        start_date=start_date,
        end_date=end_date,
        adjust=""
    )

    # 如果兜底接口也没有返回有效数据，则抛出异常交给上层处理。
    if df is None or len(df) == 0:
        raise RuntimeError(f"stock_zh_a_hist returned empty result: {symbol}")

    return standardize_stock_df(df, symbol)



def fetch_single_stock_data(symbol: str, start_date: str = START_DATE, end_date: str = END_DATE) -> pd.DataFrame:
    """抓取单只股票数据并统一成后续流程可用的格式，让后面的特征工程、训练和评估都能直接复用这一份标准化结果。"""
    # 先确认依赖无误，再启动带重试和回退策略的数据抓取流程。
    check_dependencies()
    print(f"  [Fetch] stock data (Sina first): {symbol}")

    # 按“新浪优先、东方财富兜底”的顺序准备多个抓取方案。
    fetch_methods = [
        ("stock_zh_a_daily[Sina-first]", lambda: _try_stock_daily_sina(symbol, start_date, end_date)),
        ("stock_zh_a_hist[EM-fallback]", lambda: _try_stock_hist_em(symbol, start_date, end_date)),
    ]

    errors = []
    # 依次尝试不同数据源，某个成功后就直接返回。
    for method_name, method in fetch_methods:
        try:
            print(f"    [Try] {method_name} -> {symbol}")
            # 对每个数据源都套一层重试机制，尽量降低偶发失败带来的影响。
            df = with_retry(method, max_retries=4, sleep_min=1.0, sleep_max=2.5, desc=f"{symbol} via {method_name}")
            if df is None or len(df) == 0:
                # 只有所有方案都失败时，才汇总错误信息并向上抛出异常。
                raise RuntimeError(f"{method_name} returned empty after success call: {symbol}")
            print(f"    [OK] {symbol} via {method_name}, rows={len(df)}")
            return df.copy()
        except Exception as e:
            errors.append(f"{method_name}: {repr(e)}")
            print(f"    [Fallback] {method_name} failed, trying next source...")

    raise RuntimeError(
        f"Failed to fetch stock data for {symbol}. All methods failed:\n" + "\n".join(errors)
    )


print("Data interface patched: Sina first, EM fallback.")

Data interface patched: Sina first, EM fallback.


In [7]:
run_stage1_screening_10stocks(verbose=True)

Stage-1: 10-stock architecture screening (keep LSTM + 3-day close prediction)
Screen stock count: 10
Screen architectures: ['BaseLSTM', 'FeatureAttention_LSTM', 'FeatureAttention_TCN_LSTM', 'FeatureAttention_Transformer_LSTM', 'FeatureAttention_Graph_LSTM']

################################################################################################
[SCREEN] 处理股票: 600036_招商银行 | 板块=银行
################################################################################################
  [Fetch] stock data (Sina first): 600036
    [Try] stock_zh_a_daily[Sina-first] -> 600036
    [OK] 600036 via stock_zh_a_daily[Sina-first], rows=2305

[600036_招商银行] screening | BaseLSTM

Screening tune -> BaseLSTM

  Config 1/2 -> TrainConfig(hidden_dim=64, num_layers=1, dropout=0.2, learning_rate=0.001, batch_size=32, max_epochs=50, patience=8, weight_decay=0.0001)
    Epoch [001/50] train=0.390894 val=0.034537 lr=0.001
    Epoch [002/50] train=0.088345 val=0.033835 lr=0.001
    Epoch [003/50] train=0.065

In [8]:
# =========================
# 阶段二实现：25 股全量实验 + 消融分析 + 8 组调参
# =========================

FULL_OUTER_DIR = ROOT_DIR / "full25_ablation_top_models"
safe_mkdir(FULL_OUTER_DIR)

# 每个模型使用 8 组配置做更充分的调参
FULL_SEARCH_SPACE = [
    TrainConfig(hidden_dim=64, num_layers=1, dropout=0.15, learning_rate=1e-3, batch_size=32, max_epochs=70, patience=10),
    TrainConfig(hidden_dim=64, num_layers=1, dropout=0.20, learning_rate=8e-4, batch_size=32, max_epochs=80, patience=10),
    TrainConfig(hidden_dim=64, num_layers=2, dropout=0.20, learning_rate=5e-4, batch_size=32, max_epochs=90, patience=12),
    TrainConfig(hidden_dim=96, num_layers=1, dropout=0.20, learning_rate=5e-4, batch_size=32, max_epochs=90, patience=12),
    TrainConfig(hidden_dim=96, num_layers=2, dropout=0.25, learning_rate=5e-4, batch_size=32, max_epochs=100, patience=12),
    TrainConfig(hidden_dim=128, num_layers=1, dropout=0.25, learning_rate=3e-4, batch_size=32, max_epochs=100, patience=12),
    TrainConfig(hidden_dim=128, num_layers=2, dropout=0.30, learning_rate=3e-4, batch_size=32, max_epochs=110, patience=14),
    TrainConfig(hidden_dim=96, num_layers=3, dropout=0.30, learning_rate=3e-4, batch_size=32, max_epochs=120, patience=14),
]


def select_models_from_screening(
    summary_csv: Path,
    win_csv: Path,
    min_win_rate: float = 0.25,
    max_drop_r2_vs_feat: float = 0.02,
):
    """根据阶段一汇总结果筛选阶段二保留的模型，综合考虑平均表现、胜率和消融分析的可读性。"""
    # 先读取阶段一的汇总表和胜率表，作为阶段二选模依据。
    summary_df = pd.read_csv(summary_csv)
    win_df = pd.read_csv(win_csv) if win_csv.exists() else pd.DataFrame()

    # 优先保留基线模型和标准注意力模型，保证后续消融对照完整。
    keep = set(["BaseLSTM", "FeatureAttention_LSTM"])  # 保留基线模型和标准注意力模型，便于做消融分析

    feat_row = summary_df[summary_df["arch_model"] == "FeatureAttention_LSTM"]
    feat_r2 = float(feat_row.iloc[0]["mean_overall_R2"]) if len(feat_row) == 1 else None

    # 保留在 R2 上接近 FeatureAttention 的模型
    # 如果能找到特征注意力基线，就按整体 R2 接近程度保留候选模型。
    if feat_r2 is not None:
        for _, row in summary_df.iterrows():
            arch = row["arch_model"]
            if row["mean_overall_R2"] >= feat_r2 - max_drop_r2_vs_feat:
                keep.add(arch)

    # 保留胜率达到阈值的模型
    # 再结合逐股票胜率，保留在更多股票上表现稳定的结构。
    if not win_df.empty:
        for _, row in win_df.iterrows():
            if float(row.get("win_rate", 0.0)) >= min_win_rate:
                keep.add(row["arch_model"])

    # 按当前筛选结果排除明显较差的模型
    # 同时手动剔除当前筛选阶段里明显表现偏弱的模型。
    poor_arches = set()
    for _, row in summary_df.iterrows():
        if row["arch_model"] in ["FeatureAttention_TCN_LSTM", "FeatureAttention_Transformer_LSTM"]:
            poor_arches.add(row["arch_model"])
    keep = [m for m in SCREEN_ARCH_NAMES if m in keep and m not in poor_arches]

    # 在可行时保证至少保留 3 个模型做消融分析
    # 若图结构模型具备分析价值，则补进最终候选集合里。
    if "FeatureAttention_Graph_LSTM" in summary_df["arch_model"].tolist() and "FeatureAttention_Graph_LSTM" not in keep:
        keep.append("FeatureAttention_Graph_LSTM")

    # 去重并保持原有顺序
    # 最后对模型列表去重，并保持顺序稳定，方便后续统一输出。
    ordered = []
    for m in keep:
        if m not in ordered:
            ordered.append(m)
    return ordered



def tune_model_with_space(
    model_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    input_dim: int,
    out_dim: int,
    cfg_space: List[TrainConfig],
    graph_adj_np: np.ndarray = None,
    verbose: bool = True,
):
    """在给定搜索空间内为模型寻找最优配置，适合阶段二这种需要更充分调参的正式实验。"""
    # 先准备一个变量，用来记录当前搜索空间中的最佳配置。
    best_result = None

    if verbose:
        print("\n" + "=" * 80)
        print(f"Full tune -> {model_name} | trials={len(cfg_space)}")
        print("=" * 80)

    # 按照给定配置空间逐个训练，做更充分的阶段二调参。
    for i, cfg in enumerate(cfg_space, 1):
        if verbose:
            print(f"\n  Trial {i}/{len(cfg_space)} -> {cfg}")

        # 用当前配置构造模型，并准备对应的训练和验证数据。
        model = build_screen_model(
            model_name=model_name,
            input_dim=input_dim,
            cfg=cfg,
            out_dim=out_dim,
            graph_adj_np=graph_adj_np,
        )

        train_loader, val_loader = create_train_val_loaders(
            X_train,
            y_train,
            batch_size=cfg.batch_size,
            val_ratio=VAL_RATIO_WITHIN_TRAIN,
        )

        # 运行训练流程，记录该配置在验证集上的最佳表现。
        model, history, best_val_loss = train_with_early_stopping(
            model,
            train_loader,
            val_loader,
            cfg,
            verbose=verbose,
        )

        result = {
            "config": cfg,
            "history": history,
            "best_val_loss": best_val_loss,
            "state_dict": copy.deepcopy(model.state_dict()),
        }

        # 持续比较验证损失，只保留当前表现最好的那组结果。
        if best_result is None or best_val_loss < best_result["best_val_loss"]:
            best_result = result

    return best_result



def run_single_stock_full(stock_info: Dict[str, str], selected_arches: List[str], verbose: bool = True) -> pd.DataFrame:
    """对单只股票执行阶段二全量实验，会在筛选后的模型集合上完成更充分的调参、评估和结果落盘。"""
    # 先解析当前股票的基础信息，后续日志输出和文件命名都会用到。
    symbol = stock_info["symbol"]
    name = stock_info["name"]
    sector = stock_info["sector"]
    stock_label = f"{symbol}_{name}"

    # 为当前股票创建阶段二的独立结果目录。
    stock_dir = FULL_OUTER_DIR / stock_label
    safe_mkdir(stock_dir)

    print("\n" + "#" * 100)
    print(f"[FULL] 处理股票: {stock_label} | 板块={sector}")
    print("#" * 100)

    # 抓取原始行情数据，再构造阶段二使用的特征与样本。
    raw_df = fetch_single_stock_data(symbol, START_DATE, END_DATE)
    feature_df = create_features(raw_df)

    # 若特征工程后的样本太少，则直接跳过这只股票。
    if len(feature_df) < LOOKBACK + FORECAST_HORIZONS + 30:
        raise ValueError(f"{stock_label}: valid rows too few after feature engineering ({len(feature_df)})")

    # 把连续时间序列转换成监督学习样本，并按时间顺序切分数据集。
    X_all, Y_all, target_dates_all, feature_cols = build_multi_horizon_dataset(
        feature_df,
        lookback=LOOKBACK,
        horizons=FORECAST_HORIZONS,
    )

    n_samples = len(X_all)
    train_idx, test_idx = chronological_split_indices(n_samples, TEST_RATIO)

    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
    Y_train_raw, Y_test_raw = Y_all[train_idx], Y_all[test_idx]
    dates_test = target_dates_all[test_idx]

    # 只用训练集拟合标准化器，避免测试集信息泄漏到训练阶段。
    X_train_scaled, X_test_scaled, scaler_X = fit_transform_X(X_train_raw, X_test_raw)
    Y_train_scaled, Y_test_scaled, scaler_Y = fit_transform_Y_multi(Y_train_raw, Y_test_raw)

    # 若模型需要图结构输入，这里先根据训练数据构造特征图。
    graph_adj_np = build_feature_graph_from_X(X_train_scaled, top_k=6)

    # 保存标准化器和特征列，为后续复现、分析和部署做准备。
    joblib.dump(scaler_X, stock_dir / "scaler_X.pkl")
    joblib.dump(scaler_Y, stock_dir / "scaler_Y.pkl")
    pd.Series(feature_cols, name="feature").to_csv(stock_dir / "feature_columns.csv", index=False)

    input_dim = X_train_scaled.shape[-1]
    all_rows = []

    # 依次训练阶段二保留下来的模型，并记录各自结果。
    for arch in selected_arches:
        print(f"\n[{stock_label}] full25 | {arch}")

        # 先在更大的搜索空间里调参，再恢复该模型的最佳权重。
        best_result = tune_model_with_space(
            model_name=arch,
            X_train=X_train_scaled,
            y_train=Y_train_scaled,
            input_dim=input_dim,
            out_dim=FORECAST_HORIZONS,
            cfg_space=FULL_SEARCH_SPACE,
            graph_adj_np=graph_adj_np,
            verbose=verbose,
        )

        best_cfg = best_result["config"]
        model = build_screen_model(
            model_name=arch,
            input_dim=input_dim,
            cfg=best_cfg,
            out_dim=FORECAST_HORIZONS,
            graph_adj_np=graph_adj_np,
        )
        model.load_state_dict(best_result["state_dict"])

        # 用最佳模型在测试集上预测，并计算整体与分步指标。
        preds_raw, horizon_metrics, overall_metrics = evaluate_direct_multi(
            model=model,
            X_test=X_test_scaled,
            Y_test_scaled=Y_test_scaled,
            Y_test_raw=Y_test_raw,
            scaler_Y=scaler_Y,
            batch_size=best_cfg.batch_size,
        )

        params = count_parameters(model)

        # 把预测结果、训练曲线和模型权重等文件分别落盘保存。
        prediction_matrix_to_df(dates_test, Y_test_raw, preds_raw).to_csv(
            stock_dir / f"direct_multi_{arch}_predictions.csv", index=False
        )

        torch.save(model.state_dict(), stock_dir / f"direct_multi_{arch}_best_model.pth")

        plot_training_curve(
            best_result["history"],
            f"{stock_label} - full25 - {arch} training",
            stock_dir / f"direct_multi_{arch}_training_curve.png",
        )

        plot_three_horizon_prediction(
            dates_test,
            Y_test_raw,
            preds_raw,
            f"{stock_label} - full25 - {arch}",
            stock_dir / f"direct_multi_{arch}_prediction.png",
        )

        # 将当前模型的关键指标整理进单股汇总表，便于后续总汇总。
        all_rows.append(
            metrics_row(
                symbol,
                name,
                sector,
                arch,
                params,
                n_samples,
                len(train_idx),
                len(test_idx),
                horizon_metrics,
                overall_metrics,
                asdict(best_cfg),
            )
        )

    stock_result_df = pd.DataFrame(all_rows)
    stock_result_df.to_csv(stock_dir / "stock_all_results.csv", index=False)
    return stock_result_df



def summarize_full_results(all_result_df: pd.DataFrame, save_dir: Path):
    """汇总 25 只股票的阶段二结果并输出消融统计，方便从整体上比较模型效果、误差和参数规模。"""
    # 先确保阶段二汇总目录存在，并保存完整原始结果表。
    safe_mkdir(save_dir)
    all_result_df.to_csv(save_dir / "all_25stocks_full_results.csv", index=False)

    # 按模型汇总平均 R2、MAE、参数量等核心指标。
    summary = all_result_df.groupby(["arch_model"]).agg(
        stocks=("symbol", "count"),
        mean_overall_R2=("overall_R2", "mean"),
        std_overall_R2=("overall_R2", "std"),
        mean_overall_MAE=("overall_MAE", "mean"),
        std_overall_MAE=("overall_MAE", "std"),
        mean_h1_R2=("h1_R2", "mean"),
        mean_h2_R2=("h2_R2", "mean"),
        mean_h3_R2=("h3_R2", "mean"),
        mean_params=("params", "mean"),
    ).reset_index()

    # 按整体 R2 排序，便于直接比较不同模型的整体效果。
    summary = summary.sort_values("mean_overall_R2", ascending=False)
    summary.to_csv(save_dir / "summary_25stocks_full.csv", index=False)

    # 计算相对 Base 和 FeatureAttention 的消融增益
    # 取出基线模型和特征注意力模型，作为后续消融对照。
    base_row = summary[summary["arch_model"] == "BaseLSTM"]
    feat_row = summary[summary["arch_model"] == "FeatureAttention_LSTM"]

    base_r2 = float(base_row.iloc[0]["mean_overall_R2"]) if len(base_row) == 1 else np.nan
    feat_r2 = float(feat_row.iloc[0]["mean_overall_R2"]) if len(feat_row) == 1 else np.nan

    # 计算每个模型相对两种基线的 R2 增益，形成消融结果。
    summary["delta_vs_base_r2"] = summary["mean_overall_R2"] - base_r2
    summary["delta_vs_featattn_r2"] = summary["mean_overall_R2"] - feat_r2
    summary.to_csv(save_dir / "summary_25stocks_full_with_ablation.csv", index=False)

    # 输出全局柱状图和热力图，帮助从总体上解读正式实验结果。
    plot_summary_bars(all_result_df, save_dir)
    plot_r2_heatmap(all_result_df, save_dir)

    # 同时在控制台打印一份紧凑汇总，方便快速查看最终结论。
    print("\n" + "=" * 150)
    print("25股全量汇总 + 消融结果")
    print("=" * 150)
    print(f"{'arch_model':<38}{'mean_overall_R2':<18}{'mean_overall_MAE':<18}{'delta_vs_base_r2':<18}{'delta_vs_feat_r2':<18}")
    print("=" * 150)
    for _, row in summary.iterrows():
        print(
            f"{row['arch_model']:<38}"
            f"{row['mean_overall_R2']:<18.4f}"
            f"{row['mean_overall_MAE']:<18.4f}"
            f"{row['delta_vs_base_r2']:<18.4f}"
            f"{row['delta_vs_featattn_r2']:<18.4f}"
        )
    print("=" * 150)



def run_stage2_full25_ablation(verbose: bool = True):
    """执行 25 只股票的阶段二全量实验入口，会先读取阶段一结果再开展正式训练与消融分析。"""
    # 先定位阶段一生成的汇总文件，作为阶段二选模入口。
    summary_csv = SCREEN_OUTER_DIR / "summary_10stocks_screening.csv"
    win_csv = SCREEN_OUTER_DIR / "win_rate_vs_feature_attention.csv"

    if not summary_csv.exists():
        raise FileNotFoundError(
            f"Screening summary not found: {summary_csv}. Please run stage-1 screening first."
        )

    # 根据筛选结果决定阶段二真正需要训练的模型集合。
    selected_arches = select_models_from_screening(summary_csv, win_csv)
    print("=" * 120)
    print("Stage-2: Full 25-stock run with ablation")
    print("=" * 120)
    print(f"Selected arches after pruning: {selected_arches}")
    print(f"Tuning trials per model: {len(FULL_SEARCH_SPACE)}")
    print(f"Stock count: {len(STOCK_POOL)}")

    # 保存阶段二股票池和调参空间，便于完整复现实验设置。
    pd.DataFrame(STOCK_POOL).to_csv(FULL_OUTER_DIR / "full_stock_pool.csv", index=False)
    pd.DataFrame([asdict(cfg) for cfg in FULL_SEARCH_SPACE]).to_csv(FULL_OUTER_DIR / "full_tuning_space.csv", index=False)

    all_result_list = []
    # 对 25 只股票逐只执行正式实验，单只失败不会影响整体继续运行。
    for stock_info in STOCK_POOL:
        try:
            stock_df = run_single_stock_full(stock_info=stock_info, selected_arches=selected_arches, verbose=verbose)
            all_result_list.append(stock_df)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"[Error] Stage-2 failed on {stock_info['symbol']} - {stock_info['name']}: {e}")

    # 如果没有任何股票成功完成，则直接结束并给出错误提示。
    if not all_result_list:
        print("[Error] Stage-2 failed on all stocks.")
        return

    # 只要存在有效结果，就继续做全局汇总和消融分析。
    all_result_df = pd.concat(all_result_list, ignore_index=True)
    summarize_full_results(all_result_df, FULL_OUTER_DIR)
    print(f"\nStage-2 results saved to: {FULL_OUTER_DIR.resolve()}")


# 按当前需求直接启动全量实验
run_stage2_full25_ablation(verbose=True)

Stage-2: Full 25-stock run with ablation
Selected arches after pruning: ['BaseLSTM', 'FeatureAttention_LSTM', 'FeatureAttention_Graph_LSTM']
Tuning trials per model: 8
Stock count: 25

####################################################################################################
[FULL] 处理股票: 600036_招商银行 | 板块=银行
####################################################################################################
  [Fetch] stock data (Sina first): 600036
    [Try] stock_zh_a_daily[Sina-first] -> 600036
    [OK] 600036 via stock_zh_a_daily[Sina-first], rows=2305

[600036_招商银行] full25 | BaseLSTM

Full tune -> BaseLSTM | trials=8

  Trial 1/8 -> TrainConfig(hidden_dim=64, num_layers=1, dropout=0.15, learning_rate=0.001, batch_size=32, max_epochs=70, patience=10, weight_decay=0.0001)
    Epoch [001/70] train=0.389478 val=0.022158 lr=0.001
    Epoch [002/70] train=0.054204 val=0.043538 lr=0.001
    Epoch [003/70] train=0.070643 val=0.013280 lr=0.001
    Epoch [004/70] train=0.022037 val=

In [10]:
# =========================
# 阶段三：结果解读材料生成（3 只代表股票 + 图表）
# =========================

RESULT_CSV = FULL_OUTER_DIR / "all_25stocks_full_results.csv"
SUMMARY_CSV = FULL_OUTER_DIR / "summary_25stocks_full_with_ablation.csv"
PLOT_DIR = FULL_OUTER_DIR / "selected_stock_plots"
safe_mkdir(PLOT_DIR)

all_df = pd.read_csv(RESULT_CSV)
summary_df = pd.read_csv(SUMMARY_CSV)
all_df["symbol"] = all_df["symbol"].astype(str).str.zfill(6)

model_order = ["BaseLSTM", "FeatureAttention_LSTM", "FeatureAttention_Graph_LSTM"]

# 同时根据绝对表现和模型差异度给股票打分。
stock_stat = all_df.pivot_table(index=["symbol", "name"], columns="arch_model", values="overall_R2")
stock_stat = stock_stat[[m for m in model_order if m in stock_stat.columns]].dropna().reset_index()
stock_stat["best_r2"] = stock_stat[[c for c in model_order if c in stock_stat.columns]].max(axis=1)
stock_stat["spread_r2"] = stock_stat[[c for c in model_order if c in stock_stat.columns]].max(axis=1) - stock_stat[[c for c in model_order if c in stock_stat.columns]].min(axis=1)

# 优先选择表现好且区分度明显的股票。
candidates = stock_stat[(stock_stat["best_r2"] >= 0.85) & (stock_stat["spread_r2"] >= 0.05)].copy()
if len(candidates) < 3:
    candidates = stock_stat[(stock_stat["best_r2"] >= 0.80)].copy()

candidates["select_score"] = candidates["best_r2"] + 0.9 * candidates["spread_r2"]
selected = candidates.sort_values("select_score", ascending=False).head(3).copy()
selected_symbols = selected["symbol"].tolist()

print("Selected stocks for visualization:")
print(selected[["symbol", "name", "best_r2", "spread_r2", "select_score"]].to_string(index=False))


def plot_stock_model_comparison(stock_symbol: str, stock_name: str):
    """绘制单只股票在多个模型下的预测曲线对比图，把不同结构在三个预测步长上的表现放到同一张图里展示。"""
    # 先定位当前股票在阶段二输出目录中的结果位置。
    stock_dir = FULL_OUTER_DIR / f"{stock_symbol}_{stock_name}"

    # 收集几种代表模型对应的预测结果文件路径。
    pred_files = {
        "BaseLSTM": stock_dir / "direct_multi_BaseLSTM_predictions.csv",
        "FeatureAttention_LSTM": stock_dir / "direct_multi_FeatureAttention_LSTM_predictions.csv",
        "FeatureAttention_Graph_LSTM": stock_dir / "direct_multi_FeatureAttention_Graph_LSTM_predictions.csv",
    }

    # 至少需要两份有效结果，绘出的对比图才有实际意义。
    available = {k: v for k, v in pred_files.items() if v.exists()}
    if len(available) < 2:
        print(f"[Skip] not enough prediction files: {stock_symbol}_{stock_name}")
        return None

    # 把可用结果读入内存，并以第一份结果作为日期参考。
    frames = {k: pd.read_csv(v) for k, v in available.items()}
    first_key = next(iter(frames.keys()))
    ref_df = frames[first_key]

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)

    # 按 t+1、t+2、t+3 三个预测步长分别绘制对比曲线。
    for h in range(1, 4):
        date_col = f"date_t{h}"
        true_col = f"true_t{h}"
        pred_col = f"pred_t{h}"

        dt = pd.to_datetime(ref_df[date_col])
        axes[h - 1].plot(dt, ref_df[true_col], label="真实值", linewidth=1.9)

        for model_name in ["BaseLSTM", "FeatureAttention_LSTM", "FeatureAttention_Graph_LSTM"]:
            if model_name in frames:
                axes[h - 1].plot(dt, frames[model_name][pred_col], label=model_name, linewidth=1.4)

        axes[h - 1].set_title(f"{stock_symbol}_{stock_name} - 第 t+{h} 天预测对比")
        axes[h - 1].grid(alpha=0.3)
        axes[h - 1].legend(fontsize=8)

    axes[-1].set_xlabel("日期")
    plt.tight_layout()

    # 最后把生成的对比图保存到展示目录，并返回文件路径。
    out_path = PLOT_DIR / f"{stock_symbol}_{stock_name}_3model_compare.png"
    plt.savefig(out_path, dpi=180)
    plt.close()
    return out_path


plot_records = []
for _, row in selected.iterrows():
    sym = row["symbol"]
    name = row["name"]
    out = plot_stock_model_comparison(sym, name)
    if out is not None:
        plot_records.append({
            "symbol": sym,
            "name": name,
            "plot_path": str(out),
            "best_r2": float(row["best_r2"]),
            "spread_r2": float(row["spread_r2"]),
        })

plot_df = pd.DataFrame(plot_records)
plot_df.to_csv(PLOT_DIR / "selected_stock_plot_manifest.csv", index=False)

selected_result_df = all_df[all_df["symbol"].isin(selected_symbols)].copy()
selected_result_df.to_csv(PLOT_DIR / "selected_stock_model_metrics.csv", index=False)
summary_df.to_csv(PLOT_DIR / "global_model_summary.csv", index=False)

print("\nSaved plot outputs to:")
print(PLOT_DIR.resolve())
print("Generated files:")
print("- selected_stock_plot_manifest.csv")
print("- selected_stock_model_metrics.csv")
print("- global_model_summary.csv")

Selected stocks for visualization:
symbol name  best_r2  spread_r2  select_score
300760 迈瑞医疗 0.883475   0.101899      0.975184
000338 潍柴动力 0.912926   0.063845      0.970386
002415 海康威视 0.889703   0.077239      0.959218

Saved plot outputs to:
D:\Vscode\code_html_css_vue\wzr\thesis_25stocks_direct_multi_final\full25_ablation_top_models\selected_stock_plots
Generated files:
- selected_stock_plot_manifest.csv
- selected_stock_model_metrics.csv
- global_model_summary.csv
